In [ ]:
from pathlib import Path
import re
import pandas as pd

ROOT_DIR = Path("/Users/thaddeusteh/Github/player_converter/Weekend Warriors")
RAW_DIR = ROOT_DIR / "_raw_csv"
OUTPUT_DIR = ROOT_DIR / "_outputs"
RAW_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

try:
    import tabula
except ImportError:
    tabula = None

FIELD_REGEXES = {
    "player_name": [
        r"full\s*name(?:\s*as\s*per\s*ic)?\*?\s*[:\-*]?\s*(.+)$",
        r"fullname(?:\s*as\s*per\s*ic)?\*?\s*[:\-*]?\s*(.+)$",
    ],
    "nickname": [
        r"nickname\*?\s*[:\-*]?\s*(.+)$",
    ],
    "ic_number": [
        r"(?:nric\s*no\.?|ic\s*no\.?)\*?\s*[:\-*]?\s*(.+)$",
    ],
    "email": [
        r"(?:e-?mail(?:\s*add)?|email\s*no\.?)\*?\s*[:\-*]?\s*(.+)$",
    ],
    "jersey": [
        r"jersey(?:’s|s)?\s*no\.?\*?\s*[:\-*]?\s*(.+)$",
        r"jersey\s*no\.?\*?\s*[:\-*]?\s*(.+)$",
    ],
}


def clean_text(value):
    if pd.isna(value):
        return ""
    text = str(value).replace("\u2019", "'").replace("\xa0", " ").strip()
    return re.sub(r"\s+", " ", text)


def strip_placeholder(text):
    text = clean_text(text).replace("_", " ").replace("*", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip(" :-|")


def first_nonempty(row):
    for value in row:
        text = clean_text(value)
        if text:
            return text
    return ""


def row_text(row):
    return " | ".join(clean_text(value) for value in row if clean_text(value))


def normalize(text):
    text = clean_text(text).lower().replace("’", "'")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def extract_team_name(df, fallback_name):
    for row in df.itertuples(index=False, name=None):
        line = row_text(row)
        match = re.search(r"team\s*name\s*[:\-]?\s*(.+)$", line, flags=re.I)
        if match:
            value = strip_placeholder(match.group(1))
            if value:
                return value
    return strip_placeholder(fallback_name.replace("_", " "))


def load_source_table(path):
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, header=None)
    if suffix == ".pdf":
        if tabula is None:
            raise RuntimeError(
                "tabula-py is not installed, so PDF extraction is skipped. "
                "Install tabula-py to process the Garage 94 PDF."
            )
        frames = tabula.read_pdf(str(path), pages="all", multiple_tables=True, stream=True, lattice=True)
        frames = [frame for frame in frames if frame is not None and not frame.empty]
        if not frames:
            return pd.DataFrame()
        combined = []
        for frame in frames:
            combined.append(frame)
            combined.append(pd.DataFrame([[]]))
        return pd.concat(combined, ignore_index=True)
    raise ValueError(f"Unsupported file type: {path.suffix}")


def save_raw_csv(df, source_path):
    csv_path = RAW_DIR / f"{source_path.stem}.csv"
    df.to_csv(csv_path, index=False, header=False)
    return csv_path


def get_cell(values, index):
    if index < len(values):
        value = clean_text(values[index])
        return value if value else pd.NA
    return pd.NA


def extract_value(line, patterns):
    for pattern in patterns:
        match = re.search(pattern, line, flags=re.I)
        if match:
            return strip_placeholder(match.group(1))
    return ""


def is_wide_roster(df):
    for row in df.itertuples(index=False, name=None):
        label = normalize(first_nonempty(row))
        if "jersey" not in label:
            continue
        if sum(1 for value in row[1:] if clean_text(value)) >= 2:
            return True
    return False


def parse_wide_roster(df, team_name, source_file):
    records = []
    rows = list(df.itertuples(index=False, name=None))
    start_indices = [index for index, row in enumerate(rows) if "player details" in normalize(first_nonempty(row))]
    if not start_indices:
        return records

    start_indices.append(len(rows))
    for block_start, block_end in zip(start_indices, start_indices[1:]):
        block_rows = rows[block_start + 1:block_end]
        field_values = {}
        player_count = 0

        for row in block_rows:
            label = normalize(first_nonempty(row))
            if not label:
                continue

            values = list(row[1:])
            if "jersey" in label:
                last_nonempty = -1
                for idx, value in enumerate(values):
                    if clean_text(value):
                        last_nonempty = idx
                if last_nonempty >= 0:
                    player_count = max(player_count, last_nonempty + 1)
                field_values["jersey"] = values
            elif "full name" in label or "fullname" in label:
                field_values["player_name"] = values
            elif "nickname" in label:
                field_values["nickname"] = values
            elif "nric" in label or "ic no" in label:
                field_values["ic_number"] = values
            elif "email" in label:
                field_values["email"] = values

        for index in range(player_count):
            record = {
                "player_name": get_cell(field_values.get("player_name", []), index),
                "ic_number": get_cell(field_values.get("ic_number", []), index),
                "email": get_cell(field_values.get("email", []), index),
                "team_name": team_name,
                "team_id": pd.NA,
                "jersey": get_cell(field_values.get("jersey", []), index),
                "nickname": get_cell(field_values.get("nickname", []), index),
            }
            if pd.notna(record["player_name"]) and pd.notna(record["jersey"]):
                records.append(record)

    return records


def parse_vertical_roster(df, team_name, source_file):
    records = []
    current = {
        "player_name": pd.NA,
        "ic_number": pd.NA,
        "email": pd.NA,
        "team_name": team_name,
        "team_id": pd.NA,
        "jersey": pd.NA,
        "nickname": pd.NA,
    }

    def flush():
        nonlocal current
        if pd.notna(current["player_name"]) and pd.notna(current["jersey"]):
            records.append(current.copy())
        current = {
            "player_name": pd.NA,
            "ic_number": pd.NA,
            "email": pd.NA,
            "team_name": team_name,
            "team_id": pd.NA,
            "jersey": pd.NA,
            "nickname": pd.NA,
        }

    for row in df.itertuples(index=False, name=None):
        line = row_text(row)
        normalized = normalize(line)
        if not normalized:
            continue

        if re.search(r"\*\s*\d+\.\s*player\s*\*", normalized):
            flush()
            continue

        extracted = None
        for field, patterns in FIELD_REGEXES.items():
            value = extract_value(line, patterns)
            if value:
                extracted = (field, value)
                break

        if extracted is None:
            continue

        field, value = extracted
        if field == "player_name" and pd.notna(current["player_name"]) and pd.notna(current["jersey"]):
            flush()

        current[field] = value

    flush()
    return records


source_files = [
    path for path in sorted(ROOT_DIR.rglob("*"))
    if path.is_file() and path.suffix.lower() in {".xlsx", ".xls", ".pdf"}
    and "_raw_csv" not in path.parts
    and "_outputs" not in path.parts
]

all_records = []
failed_files = []

for source_path in source_files:
    try:
        table = load_source_table(source_path)
        save_raw_csv(table, source_path)
        team_name = extract_team_name(table, source_path.parent.name)

        if is_wide_roster(table):
            records = parse_wide_roster(table, team_name, source_path)
        else:
            records = parse_vertical_roster(table, team_name, source_path)

        all_records.extend(records)
        print(f"Processed {source_path.name}: {len(records)} player rows")
    except Exception as exc:
        failed_files.append({"file": str(source_path), "error": str(exc)})
        print(f"Skipped {source_path.name}: {exc}")

players_df = pd.DataFrame(
    all_records,
    columns=["player_name", "ic_number", "email", "team_name", "team_id", "jersey", "nickname"],
)

players_df = (
    players_df
    .dropna(subset=["player_name", "jersey"], how="any")
    .drop_duplicates()
    .reset_index(drop=True)
)

players_df.to_csv(OUTPUT_DIR / "player_details.csv", index=False)

print(f"Total extracted players: {len(players_df)}")
print(f"Raw CSV folder: {RAW_DIR}")
print(f"Final CSV: {OUTPUT_DIR / 'player_details.csv'}")

if failed_files:
    print("\nFiles that still need attention:")
    pd.DataFrame(failed_files)
else:
    print("\nAll source files were processed.")

players_df

In [ ]:
output_csv = OUTPUT_DIR / "player_details.csv"
output_xlsx = OUTPUT_DIR / "player_details.xlsx"

excel_df = pd.read_csv(output_csv)
excel_df.to_excel(output_xlsx, index=False)

print(f"Excel spreadsheet created: {output_xlsx}")
excel_df